In [1]:
!pip install --upgrade torch transformers
!pip install datasets transformers accelerate

In [2]:
#load the dataset
from datasets import load_dataset

# Load your CSV dataset
data = load_dataset("csv", data_files="dataset/easy_advanced_cities.csv")
data
#cities_data = load_dataset("csv", data_files="dataset/worldcities.csv")
#cities_data

/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


DatasetDict({
    train: Dataset({
        features: ['City', 'Hard', 'Easy'],
        num_rows: 50
    })
})

In [3]:
from transformers import GPT2Tokenizer

# Preprocess the data directly from the Hugging Face DatasetDict
def preprocess_data(dataset):
    inputs = []
    outputs = []

    for example in dataset:
        city = example['City']
        easy_clue = example['Easy']
        hard_clue = example['Hard']

        # Append both easy and hard clues separately with the city as the input
        inputs.append(f"City: {city} | Generate an Easy Clue:")
        outputs.append(easy_clue)

        inputs.append(f"City: {city} | Generate a Hard Clue:")
        outputs.append(hard_clue)

    return inputs, outputs

# Assuming `data` is a DatasetDict or Dataset object
inputs, outputs = preprocess_data(data['train'])  # Use 'train', 'validation' or appropriate split

# Initialize the GPT-2 tokenizer
tokenizer = GPT2Tokenizer.from_pretrained('gpt2')
tokenizer.pad_token = tokenizer.eos_token  # Padding token for GPT-2

# Tokenize the data for GPT-2
tokenized_inputs = tokenizer(inputs, padding='max_length', truncation=True, max_length=512, return_tensors='pt')
tokenized_outputs = tokenizer(outputs, padding='max_length', truncation=True, max_length=512, return_tensors='pt')

In [4]:
import torch
from torch.utils.data import Dataset

class ClueDataset(Dataset):
    def __init__(self, input_encodings, output_encodings):
        self.input_encodings = input_encodings
        self.output_encodings = output_encodings

    def __len__(self):
        return len(self.input_encodings['input_ids'])

    def __getitem__(self, idx):
        return {
            'input_ids': self.input_encodings['input_ids'][idx],
            'attention_mask': self.input_encodings['attention_mask'][idx],
            'labels': self.output_encodings['input_ids'][idx]  # GPT-2 expects labels to be input IDs for auto-regressive tasks
        }

# Create dataset
dataset = ClueDataset(tokenized_inputs, tokenized_outputs)

In [5]:
from transformers import GPT2LMHeadModel, Trainer, TrainingArguments

# Load GPT-2 model
model = GPT2LMHeadModel.from_pretrained('gpt2')

# Training arguments
training_args = TrainingArguments(
    output_dir='./results',
    overwrite_output_dir=True,
    num_train_epochs=5,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    logging_dir='./logs',
    logging_steps=10,
    save_steps=100,
    eval_strategy="steps",
    eval_steps=100,
    learning_rate=5e-5
)

# Define Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset,  # We are using the same dataset for simplicity
    eval_dataset=dataset,   # You should split train and eval data for better performance
)

# Start training
trainer.train()

Step,Training Loss,Validation Loss
100,0.440800,0.423995
200,0.368100,0.400273


TrainOutput(global_step=250, training_loss=0.5505480937957764, metrics={'train_runtime': 323.9356, 'train_samples_per_second': 1.544, 'train_steps_per_second': 0.772, 'total_flos': 130646016000000.0, 'train_loss': 0.5505480937957764, 'epoch': 5.0})

In [6]:
import torch
from transformers import GPT2Tokenizer, GPT2LMHeadModel

# Load the model and tokenizer
tokenizer = GPT2Tokenizer.from_pretrained("gpt2")
model = GPT2LMHeadModel.from_pretrained("gpt2")

# Ensure we use the EOS token as the PAD token
tokenizer.pad_token = tokenizer.eos_token

# Set everything to run on CPU
device = torch.device("cpu")
model.to(device)

# Function to generate clues
def generate_clue(city_name, clue_type="Easy"):
    model.eval()
    input_text = f"City: {city_name} | Generate a {clue_type} Clue:"
    
    # Tokenize and move input to CPU
    input_ids = tokenizer(input_text, return_tensors='pt').input_ids.to(device)
    
    # Generate clue
    generated_ids = model.generate(input_ids, max_new_tokens=50, pad_token_id=tokenizer.eos_token_id)
    
    # Decode the generated tokens
    return tokenizer.decode(generated_ids[0], skip_special_tokens=True)

# Example usage
city = "Berlin"
generated_easy_clue = generate_clue(city, clue_type="Easy")
generated_hard_clue = generate_clue(city, clue_type="Hard")

print(f"Generated Easy Clue: {generated_easy_clue}")
print(f"Generated Hard Clue: {generated_hard_clue}")

The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


Generated Easy Clue: City: Berlin | Generate a Easy Clue: The German Government's Secret Plan to Create a New World Order

The German government has been working on a plan to create a new world order, but it's not clear how it will work.

The plan is to create a new world
Generated Hard Clue: City: Berlin | Generate a Hard Clue: The German Army's Secret History

The German Army's Secret History: The German Army's Secret History

The German Army's Secret History: The German Army's Secret History

The German Army's Secret History: The German Army's Secret
